# 문제 2-2 : 영화 리뷰 감정 분석기

영화 리뷰 텍스트를 입력받아 감정을 **"긍정"**, **"부정"**, **"보통"** 중 하나로 분류한다.

**출력 파서**: `EnumOutputParser`

**요구사항 체크**
- [x] `EnumOutputParser` 와 `Enum` 클래스 사용
- [x] 여러 개의 테스트 리뷰로 검증 (문제의 테스트 리뷰 3개 + 추가 3개, 기대 감정과 비교)
- [x] 감정 분류 결과를 깔끔하게 출력
- [x] (선택) 분석 실패 시를 대비해 `OutputFixingParser` 와 `EnumOutputParser` 함께 사용 (섹션 7)

## 1. 환경 변수 로드

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정 (.env 는 프로젝트 루트에 있음)
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 전체는 출력하지 않고 설정 여부만 확인
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

## 2. LLM 설정 (ChatOpenAI)
분류 작업이라 결과가 매번 흔들리지 않도록 `temperature=0` 으로 설정한다.

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq"(무료) 또는 "openai" — 이 값만 바꾸면 아래 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# Groq는 OpenAI 호환 API라서 ChatOpenAI에 base_url만 바꿔서 사용
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",
    },
}


def get_llm(temperature: float = 0.7, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자의 ChatOpenAI 인스턴스를 생성합니다."""
    config = LLM_CONFIGS[provider or PROVIDER]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {(provider or PROVIDER).upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(**config, temperature=temperature, max_retries=3)


llm = get_llm(temperature=0)
print(f"{PROVIDER} / {llm.model_name}")

## 3. 감정 `Enum` 클래스 정의
Enum 의 **값(value)** 이 LLM이 답해야 하는 문자열이 된다.

In [ ]:
from enum import Enum


class Sentiment(Enum):
    """영화 리뷰 감정"""
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "보통"


print([s.value for s in Sentiment])

## 4. 출력 파서 생성 — `EnumOutputParser`
- `get_format_instructions()` : Enum 값 중 하나만 고르라는 지시문을 만든다
- `parse()` : LLM 답변 문자열을 `Sentiment` 멤버로 바꾼다. Enum 값에 없는 답이면 `OutputParserException` 발생

> LangChain 1.x 에서는 `EnumOutputParser` 가 `langchain-classic` 패키지로 옮겨졌다.
> `ImportError` 가 나면 터미널에서 `uv add langchain-classic` 실행 후 커널을 재시작한다.

In [ ]:
try:
    # LangChain 1.x : EnumOutputParser 등 기존 출력 파서는 langchain-classic 패키지에 있음
    from langchain_classic.output_parsers import EnumOutputParser
except ImportError:
    # LangChain 0.3.x 이하
    from langchain.output_parsers import EnumOutputParser

output_parser = EnumOutputParser(enum=Sentiment)

# 프롬프트에 넣을 형식 지시문
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

# parse() 동작 확인: 문자열 → Enum 멤버
print(repr(output_parser.parse("부정")))

## 5. PromptTemplate 작성 & LCEL 체인 연결: `prompt | llm | output_parser`

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """당신은 영화 리뷰 감정 분석가입니다.
아래 영화 리뷰를 읽고 감정을 "긍정", "부정", "보통" 중 하나로 분류하세요.
- 긍정: 영화를 칭찬하거나 만족을 표현함
- 부정: 영화를 비판하거나 불만을 표현함
- 보통: 좋지도 나쁘지도 않은 중립적인 평가

다른 설명 없이 단어 하나만 답하세요.

리뷰: {review}

{format_instructions}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["review"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | llm | output_parser

result = chain.invoke({"review": "이 영화 정말 재미없어요. 시간 낭비였습니다."})
print(type(result))  # str 이 아니라 Sentiment(Enum) 객체
print(result, "→", result.value)

## 6. 여러 개의 테스트 리뷰로 검증
문제에서 준 테스트 리뷰 3개 + 직접 추가한 리뷰 3개에 **기대 감정**을 붙여 두고, 분류 결과와 비교한다.

In [ ]:
test_reviews = [
    # (리뷰, 기대 감정) — 문제에서 주어진 테스트 리뷰
    ("이 영화 정말 재미없어요. 시간 낭비였습니다.", Sentiment.NEGATIVE),
    ("배우들의 연기가 훌륭하고 스토리도 감동적이었어요!", Sentiment.POSITIVE),
    ("그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.", Sentiment.NEUTRAL),
    # 추가 테스트 리뷰
    ("영상미가 압도적이고 OST가 계속 귀에 맴돌아요. 극장에서 두 번 봤습니다.", Sentiment.POSITIVE),
    ("개연성이 너무 없고 결말이 허무해서 실망했어요.", Sentiment.NEGATIVE),
    ("킬링타임용으로 볼 만은 한데 기억에 남는 장면은 딱히 없네요.", Sentiment.NEUTRAL),
]

In [ ]:
from langchain_core.exceptions import OutputParserException

EMOJI = {Sentiment.POSITIVE: "😊", Sentiment.NEGATIVE: "😞", Sentiment.NEUTRAL: "😐"}


def evaluate(target_chain, reviews) -> None:
    """리뷰마다 감정을 분류해 출력하고, 기대 감정과 일치하는지 검증"""
    correct = 0
    for i, (review, expected) in enumerate(reviews, 1):
        print(f"[리뷰 {i}] {review}")
        try:
            sentiment = target_chain.invoke({"review": review})
        except OutputParserException as e:
            # LLM 답변이 "긍정/부정/보통" 중 하나가 아니면 EnumOutputParser 가 예외를 던짐
            print(f"   → ⚠️ 분석 실패: {str(e).splitlines()[0]}")
            print("-" * 70)
            continue
        is_match = sentiment == expected
        correct += is_match
        print(f"   → 감정: {EMOJI[sentiment]} {sentiment.value}   (기대: {expected.value} {'✅' if is_match else '❌'})")
        print("-" * 70)
    print(f"정확도: {correct}/{len(reviews)}")


evaluate(chain, test_reviews)

## 7. (선택) `OutputFixingParser` + `EnumOutputParser` — 분석 실패 대비

LLM이 `"이 리뷰는 부정적인 감정을 담고 있습니다."` 처럼 형식에 맞지 않게 답하면 `EnumOutputParser` 는 `OutputParserException` 을 던진다.
`OutputFixingParser` 는 이때 **원래 형식 지시문 + 잘못된 답변 + 에러 메시지**를 LLM에게 다시 보내 형식에 맞게 고친 뒤 재파싱한다.

In [ ]:
try:
    from langchain_classic.output_parsers import OutputFixingParser
except ImportError:
    from langchain.output_parsers import OutputFixingParser

# EnumOutputParser 를 감싸서, 파싱에 실패하면 llm 으로 답변을 고치게 함
fixing_parser = OutputFixingParser.from_llm(parser=output_parser, llm=llm)

bad_output = "이 리뷰는 부정적인 감정을 담고 있습니다."  # 형식에 맞지 않는 LLM 답변 예시

# 1) EnumOutputParser 단독 → 실패
try:
    output_parser.parse(bad_output)
except OutputParserException as e:
    print("EnumOutputParser 단독 :", str(e).splitlines()[0])

# 2) OutputFixingParser → LLM 이 형식을 고친 뒤 다시 파싱해서 성공
fixed = fixing_parser.parse(bad_output)
print("OutputFixingParser   :", repr(fixed))

In [ ]:
# 체인의 마지막 파서만 fixing_parser 로 교체 → 같은 테스트 리뷰로 다시 검증
fixing_chain = prompt | llm | fixing_parser

evaluate(fixing_chain, test_reviews)

## 8. (선택) 직접 리뷰 입력해 보기

In [ ]:
review = input("영화 리뷰를 입력하세요: ").strip()

if review:
    sentiment = fixing_chain.invoke({"review": review})
    print(f"리뷰: {review}")
    print(f"감정: {EMOJI[sentiment]} {sentiment.value}")
else:
    print("리뷰가 입력되지 않았습니다.")